# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [4]:
import re
import Bio.SeqIO
import Bio.Seq

seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

def HA_or_NA(ind_read, downstream="AGGCGGCCGC"):

    #make downstream in same orientation to input
    downstream_seq = Bio.Seq.Seq(downstream)
    upstream = downstream_seq.reverse_complement()

    HA = Bio.Seq.Seq("CCGGATTTGCATATAATGATGCACCAT")
    HA_revcomp = HA.reverse_complement()
    
    NA = Bio.Seq.Seq("CACGATAGATAAATAATAGTGCACCAT")
    NA_revcomp = NA.reverse_complement()

    #will only pull barcodes with correct letters and of at least 16 bases upstream of correct correect
    gene_pattern = re.compile(f"{str(upstream)}(?P<gene>[ATCGNatcgn]{{{len(str(NA))}}})")
    if type(ind_read) != str:
        ind_read = ind_read.seq
    if gene_pattern.search(str(ind_read)) != None:
        gene = gene_pattern.search(str(ind_read)).group("gene")
        if gene == NA_revcomp:
            return "NA"
        elif gene == HA_revcomp:
            return "HA"
        else:
            return None
    else:
        return None
        

test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", "NA", "positive: clean NA read"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATATTTATCTATCGTG", None , "negative: deletion in NA"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTTATCTATCGTG", None, "negative: insertion in NA"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCACTATTATTTATAAATCGTG", None, "negative: mutation in NA"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG", "HA", "positive: clean HA read"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATCAAATCCGG", None, "negative: deletion in HA"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCAAAATCCGG", None, "negative: insertion in HA"),
    ("TTTTTTTTTTTTTTTTGCGGCCGCCTATGGTGCATCATTATATGCTTTTCCGG", None, "negative: mutation in HA")
]

def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

run_tests(HA_or_NA, test_cases)

NA_count = 0
HA_count = 0
unmatched = []
for read in seqreads:
    gene = HA_or_NA(read, downstream="AGGCGGCCGC")
    if gene == None:
        unmatched.append(read.id)
        print(f"Unmatched read: {read.id} : {read.seq}")
    elif gene == "NA":
        NA_count += 1
    elif gene == "HA":
        HA_count += 1

print ()
print (f'There are {NA_count} NA reads.')
print (f'There are {HA_count} HA reads.')
print (f'There are {len(unmatched)} reads that match neither.')



PASS  positive: clean NA read
PASS  negative: deletion in NA
PASS  negative: insertion in NA
PASS  negative: mutation in NA
PASS  positive: clean HA read
PASS  negative: deletion in HA
PASS  negative: insertion in HA
PASS  negative: mutation in HA
Unmatched read: HISEQ:621:HMJGNBCX2:1:1101:14188:2196 : TCAGCGATGAATGTGGGCGGCCGCCTATGTTGCATCATTATATGCAAATCCGGCATTGCAAGGAGCCGTTGGAACACATAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAAAGCAAAGCACGCAAAACACAACCATCAGATGAAAGAAGACACTAACCATCCGACCCAACCTCATGCTATCACCAACTACCATTCATCTGA
Unmatched read: HISEQ:621:HMJGNBCX2:1:1101:1037:3030 : TTGTCGGAAGGTTAAGGCGGCCGCCTATGGTGCATCATTATATGCAAATCNGGCATTGCAAGGAGCCGTTGGAACACATAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCNGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAATCACACATTAATAATTACAATACCTGACGGGTATTGTTCACACTACCACGCTTGCATGTTGTGTTCGATATTCAGGTCTATATCTCTCAGA
Unmatched read: HISEQ:621:HMJGNBCX2:1:1101:1034:6241 : TTCTTGAGGATCGTCAGCGGCCGCCTATGGTGCACTATTATTTATCTATCNTGAAAGGGAGTTCTGCTCCATCAGGCCAAGATC

In [5]:
print(HA_or_NA("CCGCGGGGATGCGACGGCCGCCTATGGTGCATCATTATATGCAAATCCGGCATTGCAAGGAGCCGTTTGAACACATAGATCGGAAGGGCACAAGTCTGAACTCCGGTCACATGTCAGAATCTCGTATGCCGTCTTCCGCGTGTAAATACAAATCACTGGGTAAGGGCATTCACCAGATTTTTGGAGCAGCCTTCAAATCACAGTTTGGAGGAATGTCCTGGTTCTCACAGATCCTCATAGGCACGCAGCT"))

None


In [ ]:
print(Bio.Seq.Seq("TTAATGTCGGGTCGGGGCCGCCTATGGTGCATCATTATATGCAAATCCGGCATTGCAAGGAGCCGTTGGAACACATAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAAAACACTACATCAATTATATTATAACCACTTTGATAATGTAAGGACAACCCACCTGGAACATCTCGACCGAGTCTTTGCTAGCACTTTTAAGAG").reverse_complement())



CTCTTAAAAGTGCTAGCAAAGACTCGGTCGAGATGTTCCAGGTGGGTTGTCCTTACATTATCAAAGTGGTTATAATATAATTGATGTAGTGTTTTTTTTTTTTTTTTTCAAGCAGAAGACGGCATACGAGATTCTGACATGTGACTGGAGTTCAGACGTGTGCTCTTCCGATCTATGTGTTCCAACGGCTCCTTGCAATGCCGGATTTGCATATAATGATGCACCATAGGCGGCCCCGACCCGACATTAA


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [20]:
# your code here...
def read_barcode(ind_read, bclen=16, upstream="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.
    
    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """
    # your code here...

    #make upstream in same orientation to input
    upstream_seq = Bio.Seq.Seq(upstream)
    downstream = upstream_seq.reverse_complement()


    #will only pull barcodes with correct letters and of at least 16 bases upstream of correct correect
    barcode_pattern = re.compile(f"(?P<barcode>[ATGCNatcgn]{{{bclen}}}){str(downstream)}")
    if type(ind_read) != str:
        ind_read = ind_read.seq
        
    if barcode_pattern.search(str(ind_read)) != None:
        barcode = barcode_pattern.search(str(ind_read)).group("barcode")
    else:
        return None

    upstream_coord = 0
    for i in range(len(str(ind_read))):
        if str(ind_read[i:i+len(str(downstream))]) == str(downstream):
            upstream_coord = i
    #eliminates long barcodes
    if barcode != str(ind_read[:upstream_coord]):
        return None
    else:
        barcode_seq = Bio.Seq.Seq(barcode)
        barcode_fwd = barcode_seq.reverse_complement()
        return str(barcode_fwd.upper())

NA_barcode_counts = {}
NA_unmatched = []

HA_barcode_counts = {}
HA_unmatched = []


for read in seqreads:
    gene = HA_or_NA(read, downstream="AGGCGGCCGC")
    if gene == "NA":
        barcode = read_barcode(read, bclen=16, upstream="AGGCGGCCGC")
        if barcode == None:
            NA_unmatched.append(read.id)
        else:
            if barcode in NA_barcode_counts:
                NA_barcode_counts[barcode] = NA_barcode_counts[barcode] + 1
            else:
                NA_barcode_counts[barcode] = 1
    elif gene == "HA":
        barcode = read_barcode(read, bclen=16, upstream="AGGCGGCCGC")
        if barcode == None:
            HA_unmatched.append(read.id)
        else:
            if barcode in HA_barcode_counts:
                HA_barcode_counts[barcode] = HA_barcode_counts[barcode] + 1
            else:
                HA_barcode_counts[barcode] = 1


print (f'There are {len(NA_unmatched)} NA reads without a valid barcode.')
print (f'There are {len(HA_unmatched)} HA reads without a valid barcode.')


There are 63 NA reads without a valid barcode.
There are 70 HA reads without a valid barcode.


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [21]:
# your code here...

max_NA_barcode, max_NA_barcode_count = max(NA_barcode_counts.items(), key=lambda item: item[1])

max_HA_barcode, max_HA_barcode_count = max(HA_barcode_counts.items(), key=lambda item: item[1])

print (f'For NA: Barcode {max_NA_barcode} has the most counts with {max_NA_barcode_count}.')
print (f'For HA: Barcode {max_HA_barcode} has the most counts with {max_HA_barcode_count}.')



For NA: Barcode ACCAGTTCTCCCCGGG has the most counts with 152.
For HA: Barcode CCCGACCCGACATTAA has the most counts with 155.
